# Generate behavioral_signals.csv

This notebook creates monthly behavioral signals per customer for Jan 2022 → Dec 2023 using `data/customer_profile.csv`, `data/loan_details.csv`, and `data/repayment_behavior.csv` if available.
It simulates inflows, outflows, closing balances, volatility and early‑warning flags aligned with Stage‑3 rules.

# Generate behavioral_signals.csv

This notebook creates monthly behavioral signals per customer for Jan 2022 → Dec 2023 using `data/customer_profile.csv`, `data/loan_details.csv`, and `data/repayment_behavior.csv` if available.
It simulates inflows, outflows, closing balances, volatility and early‑warning flags aligned with Stage‑3 rules.

In [17]:
import numpy as np
import pandas as pd
from pathlib import Path
import random

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

In [18]:
# Read inputs
cust_path = Path('../data/customer_profile.csv')
loan_path = Path('../data/loan_details.csv')
rep_path = Path('../data/repayment_behavior.csv')
if not cust_path.exists():
    raise FileNotFoundError('Run notebooks/01_generate_customers.ipynb first to create customer_profile.csv')
cust = pd.read_csv(cust_path)
loans = pd.read_csv(loan_path) if loan_path.exists() else pd.DataFrame()
repay = pd.read_csv(rep_path) if rep_path.exists() else pd.DataFrame()

# months range: Jan 2022 to Dec 2023 (24 months)
months = pd.date_range('2022-01-01', '2023-12-31', freq='MS')

print('Customers:', len(cust), 'Loans:', len(loans), 'Repayment rows:', len(repay))

Customers: 10000 Loans: 12000 Repayment rows: 217383


In [19]:
# Precompute per-loan default info (if loan_outcome exists)
loan_defaults = {}
if Path('../data/loan_outcome.csv').exists():
    out = pd.read_csv(Path('../data/loan_outcome.csv'))
    for _, r in out.iterrows():
        loan_defaults[r['loan_id']] = {'default_flag': int(r.get('default_flag',0)) if not pd.isna(r.get('default_flag',0)) else 0, 'default_month': int(r['default_month']) if not pd.isna(r['default_month']) else None}

# Precompute customer-level lookup maps so the monthly loop stays fast
customer_loan_months = {}
customer_has_default = {}
if not loans.empty:
    loans_local = loans.copy()
    loans_local['orig_month'] = pd.to_datetime(loans_local['origination_date']).dt.strftime('%Y-%m')
    for cid, grp in loans_local.groupby('customer_id'):
        customer_loan_months[cid] = set(grp['orig_month'].dropna().tolist())
        customer_has_default[cid] = any(loan_defaults.get(lid, {}).get('default_flag', 0) == 1 for lid in grp['loan_id'].tolist())

# Build repayment lookup: payments per customer-month
payments_lookup = {}
if not repay.empty:
    repay_local = repay[['customer_id', 'due_date', 'payment_status', 'amount_paid']].copy()
    repay_local['due_month'] = pd.to_datetime(repay_local['due_date']).dt.strftime('%Y-%m')
    repay_local['missed_flag'] = (repay_local['payment_status'] == 'Missed').astype('int8')
    agg = (
        repay_local.groupby(['customer_id', 'due_month'], sort=False, observed=True)
        .agg(missed_count=('missed_flag', 'sum'), emi_paid=('amount_paid', 'sum'))
        .reset_index()
    )
    for row in agg.itertuples(index=False):
        payments_lookup[(row.customer_id, row.due_month)] = {'missed_count': int(row.missed_count), 'emi_paid': float(row.emi_paid)}

# Helper: credit sensitivity
def credit_sensitivity(score):
    try:
        s = float(score)
    except Exception:
        return 1.0
    if s < 550:
        return 1.5
    if s < 650:
        return 1.2
    return 0.8

def assign_volatility(score):
    if score < 30:
        return 'Low'
    elif score < 60:
        return 'Medium'
    else:
        return 'High'

rows = []
prev_outflow = {}  # track previous month's outflow per customer for shock detection
sig_counter = 1
for idx, c in cust.iterrows():
    cid = c['customer_id']
    seg = c.get('employment_type', '')
    base_income = float(c.get('monthly_income', 0))
    # baseline balance
    balance = base_income * np.random.uniform(0.5,2.0) if base_income>0 else np.random.uniform(500,5000)
    vol_base = 0.05 if seg=='Salaried' else (0.15 if seg=='Gig Worker' else 0.1)
    has_defaulted = customer_has_default.get(cid, False)
    if has_defaulted:
        digits = ''.join([ch for ch in cid if ch.isdigit()])
        seed_val = int(digits[-6:]) if len(digits)>=1 else idx
        rng = np.random.RandomState(seed_val + 7)
        default_cal_month = int(rng.randint(1,25))
    else:
        default_cal_month = None
    loan_month_set = customer_loan_months.get(cid, set())

    for m_idx, d in enumerate(months, start=1):
        ym = d.strftime('%Y-%m')
        # inflow
        noise = np.random.normal(0, vol_base)
        monthly_inflow = max(0.0, base_income * (1 + noise))
        inflow_drop_flag = 0
        if default_cal_month is not None and (default_cal_month - m_idx) in (1,2):
            monthly_inflow *= np.random.uniform(0.25,0.6)
            inflow_drop_flag = 1
        # outflow
        key = (cid, ym)
        emi_paid = payments_lookup.get(key, {}).get('emi_paid', None) if payments_lookup else None
        if emi_paid is not None and emi_paid>0:
            monthly_outflow = emi_paid + np.random.uniform(0.3,0.9)*monthly_inflow
        else:
            spend_frac = 0.5 if seg=='Salaried' else (0.7 if seg=='Gig Worker' else 0.6)
            monthly_outflow = monthly_inflow * np.random.uniform(max(0.1,spend_frac-0.1), spend_frac+0.2)
        # closing balance and volatility
        closing_balance = balance + monthly_inflow - monthly_outflow
        balance_volatility_score = abs(closing_balance - balance) / (abs(balance) + 1e-6) * 100
        balance_volatility = assign_volatility(balance_volatility_score)
        # spending shock: compare to previous outflow for this customer
        prev = prev_outflow.get(cid, None)
        spending_shock_flag = 1 if (prev is not None and prev>0 and monthly_outflow / (prev+1e-6) > 1.5) else 0
        missed = payments_lookup.get(key, {}).get('missed_count', 0) if payments_lookup else 0
        upi_tx = int(np.random.poisson(max(1, monthly_inflow/5000)))
        new_loan_flag = 1 if ym in loan_month_set else 0
        cs = credit_sensitivity(c.get('credit_score', None))
        score = 0
        score += min(50, balance_volatility_score * cs)
        score += min(30, missed * 10)
        score += 20 * inflow_drop_flag
        score = min(100, int(score))
        rows.append({
            'signal_id': f'SIG{str(sig_counter).zfill(8)}',
            'customer_id': cid,
            'month': ym,
            'monthly_inflow': round(monthly_inflow,2),
            'monthly_outflow': round(monthly_outflow,2),
            'closing_balance': round(closing_balance,2),
            'balance_volatility': balance_volatility,
            'inflow_drop_flag': int(inflow_drop_flag),
            'spending_shock_flag': int(spending_shock_flag),
            'missed_bill_payments': int(missed),
            'upi_transaction_count': int(upi_tx),
            'new_loan_flag': int(new_loan_flag),
            'stress_score': int(score)
        })
        sig_counter += 1
        prev_outflow[cid] = monthly_outflow
        balance = closing_balance

# write out
sig_df = pd.DataFrame(rows)
if sig_df.empty:
    print('No signals generated')
else:
    out_path = Path('../data/behavioral_signals.csv')
    sig_df.to_csv(out_path, index=False)
    print(f'Wrote {len(sig_df)} behavioral signal rows to: {out_path}')


Wrote 240000 behavioral signal rows to: ..\data\behavioral_signals.csv
